## Syncronous code 


In [1]:
import time


In [7]:
def fetch_whether_data():
    print("Fetching weather data...")
    time.sleep(2)  # Simulate a delay in fetching data
    print("Weather data fetched.")


def fetch_news_data():
    print("Fetching news data...")
    time.sleep(3)  # Simulate a delay in fetching data
    print("News data fetched.")


def main():
    start_time = time.time()

    fetch_whether_data()
    fetch_news_data()

    end_time = time.time()a
    print(f"Total time taken: {end_time - start_time:.2f} seconds")

main()

Fetching weather data...
Weather data fetched.
Fetching news data...
News data fetched.
Total time taken: 5.00 seconds


In [8]:
import asyncio

In [11]:



async def fetch_whether_data():
    print("Fetching weather data...")
    await asyncio.sleep(2)  # Simulate a delay in fetching data
    print("Weather data fetched.")


async def fetch_news_data():
    print("Fetching news data...")
    await asyncio.sleep(3)  # Simulate a delay in fetching data
    print("News data fetched.")


# def main():
#     start_time = time.time()

#     fetch_whether_data()
#     fetch_news_data()

#     end_time = time.time()
#     print(f"Total time taken: {end_time - start_time:.2f} seconds")


async def main():
    start_time = time.time()

    await asyncio.gather(
        fetch_whether_data(),
        fetch_news_data()
    )

    end_time = time.time()
    print(f"Total time taken: {end_time - start_time:.2f} seconds")

# if __name__ == "__main__":
#     asyncio.run(main())         this fails in jupyter notebook, because it tries to create a new event loop while one is already running.
#  Instead, we can use the following approach:

await main()

Fetching weather data...
Fetching news data...
Weather data fetched.
News data fetched.
Total time taken: 3.01 seconds


In [14]:
import asyncio
import time

async def task1():
    print("Task 1 started")
    await asyncio.sleep(2)
    print("Task 1 finished")

async def task2():
    print("Task 2 started")
    await asyncio.sleep(3)
    print("Task 2 finished")

# if __name__ == "__main__":
#     asyncio.run(main())         this fails in jupyter notebook, because it tries to create a new event loop while one is already running.
#  Instead, we can use the following approach:



await asyncio.gather(task1(), task2())

Task 1 started
Task 2 started
Task 1 finished
Task 2 finished


[None, None]

## Pydantic: How to Use It

Pydantic uses Python type annotations to parse and validate data at runtime. It is especially useful when data enters your program from an API, configuration file, form, or async task.

### A typical workflow

1. **Define a model** by inheriting from `BaseModel`. Annotations describe the expected fields and types.
2. **Add rules** with `Field`, such as numeric bounds, string lengths, or defaults.
3. **Validate incoming data** with `ModelName.model_validate(raw_data)`. If it does not match the model, Pydantic raises `ValidationError`.
4. **Use the validated object** through its attributes, such as `report.city`.
5. **Serialize it** with `report.model_dump()` for a Python dictionary or `report.model_dump_json()` for JSON text.

For example, `nickname: str | None = None` means the field is optional and defaults to `None`; `retries: int = 3` provides a default value. Use `Field(ge=0)` to require a number to be greater than or equal to zero.

Pydantic may convert compatible input values to the declared type. Use strict validation when conversion is not acceptable. Validation checks the data shape and declared rules; it does not replace business logic, authorization, or other application checks.

The examples below use Pydantic v2.

In [1]:
from pydantic import BaseModel, Field, ValidationError


class WeatherReport(BaseModel):
    city: str
    temperature_c: float = Field(ge=-90, le=60)
    conditions: str = Field(min_length=1)


payload = {
    "city": "Reykjavik",
    "temperature_c": 7,
    "conditions": "Cloudy",
}

report = WeatherReport.model_validate(payload)
print(report)
print(report.model_dump())

try:
    WeatherReport.model_validate({"city": "", "temperature_c": 120, "conditions": ""})
except ValidationError as error:
    print("Invalid input:")
    print(error)

city='Reykjavik' temperature_c=7.0 conditions='Cloudy'
{'city': 'Reykjavik', 'temperature_c': 7.0, 'conditions': 'Cloudy'}
Invalid input:
2 validation errors for WeatherReport
temperature_c
  Input should be less than or equal to 60 [type=less_than_equal, input_value=120, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
conditions
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


### Validate an Async Result

An async function can return untrusted raw data. Validate that result before using it elsewhere in your program.

In [5]:
import asyncio


async def fetch_weather_payload():
    await asyncio.sleep(0.1)
    return {
        "city": "Oslo",
        "temperature_c": 4.5,
        "conditions": "Light snow",
    }


raw_payload = await fetch_weather_payload()
validated_report = WeatherReport.model_validate(raw_payload)
print(validated_report)

city='Oslo' temperature_c=4.5 conditions='Light snow'


In [6]:
def add_patient_data(patient_id: int, name: str, age: int, conditions: list[str]):
    if type(patient_id) is not int or patient_id <= 0:
        raise ValueError("Patient ID must be a positive integer.")
    if type(name) is not str or not name.strip():
        raise ValueError("Name must be a non-empty string.")
    if type(age) is not int or age < 0:
        raise ValueError("Age must be a non-negative integer.")
    if type(conditions) is not list or not all(isinstance(cond, str) for cond in conditions):
        raise ValueError("Conditions must be a list of strings.")
    
    return {
        "patient_id": patient_id,
        "name": name,
        "age": age,
        "conditions": conditions,
    }


def update_patient_data(patient_data: dict, name: str = None, age: int = None, conditions: list[str] = None):
    if name is not None:
        if type(name) is not str or not name.strip():
            raise ValueError("Name must be a non-empty string.")
        patient_data["name"] = name
    if age is not None:
        if type(age) is not int or age < 0:
            raise ValueError("Age must be a non-negative integer.")
        patient_data["age"] = age
    if conditions is not None:
        if type(conditions) is not list or not all(isinstance(cond, str) for cond in conditions):
            raise ValueError("Conditions must be a list of strings.")
        patient_data["conditions"] = conditions
    
    return patient_data


In [9]:
data = add_patient_data(1, "John Doe", 30, ["Diabetes", "Hypertension"])
print(data)
# error cases
try:
    add_patient_data(-1, "Jane Doe", 25, ["Asthma"])
except ValueError as e:
    print(e)



{'patient_id': 1, 'name': 'John Doe', 'age': 30, 'conditions': ['Diabetes', 'Hypertension']}
Patient ID must be a positive integer.


In [13]:
from typing import Optional


class PatientData(BaseModel):

    # patient_id, name, age, and married are required fields
    patient_id: int = Field(gt=0)
    name: str = Field(min_length=1)
    age: int = Field(ge=0)

    # married is a boolean field with a default value of False
    married: bool = Field(default=False)

    # allergies and conditions are optional fields with default empty lists
    # can we use optional fields here? Yes, we can use optional fields with default values.
    medications: Optional[list[str]] = None
    
    allergies: list[str] = Field(default_factory=list)
    conditions: list[str] = Field(default_factory=list)


def add_patient_data(patient_id: int, name: str, age: int, married: bool, allergies: list[str], conditions: list[str]):

    if type(patient_id) is not int or patient_id <= 0:
        raise ValueError("Patient ID must be a positive integer.")
    if type(name) is not str or not name.strip():
        raise ValueError("Name must be a non-empty string.")
    if type(age) is not int or age < 0:
        raise ValueError("Age must be a non-negative integer.")
    if type(married) is not bool:
        raise ValueError("Married must be a boolean value.")
    if type(allergies) is not list:
        raise ValueError("Allergies must be a list of strings.")
    if type(conditions) is not list:
        raise ValueError("Conditions must be a list of strings.")

    patient_data = {
        "patient_id": patient_id,
        "name": name,
        "age": age,
        "married": married,
        "allergies": allergies,
        "conditions": conditions,
    }
    return PatientData.model_validate(patient_data)


data  = add_patient_data(1, "John Doe", 30, False, ["Peanuts"], ["Diabetes", "Hypertension"])

print("added " + str(data))



added patient_id=1 name='John Doe' age=30 married=False medications=None allergies=['Peanuts'] conditions=['Diabetes', 'Hypertension']


### Required and Optional Fields

In a Pydantic model, a field is **required** when it has no default value. In `PatientData`, `patient_id`, `name`, and `age` are required: omitting any of them causes validation to fail.

A field is **optional to provide** when it has a default. Here, `married` defaults to `False`, and `allergies` and `conditions` default to empty lists. `default_factory=list` creates a fresh list for each model instance, which is the right way to use mutable defaults.

Optional-to-provide and nullable are different ideas:

- `middle_name: str | None` accepts `None`, but is still required unless it has a default.
- `middle_name: str | None = None` can be omitted and can also be set to `None`.
- `nickname: str = "Guest"` can be omitted, but if supplied must be a string.

Because `PatientData` has defaults, validating a dictionary without `married`, `allergies`, or `conditions` fills them in automatically. The `add_patient_data()` helper above still requires those arguments in its function signature; model defaults apply when you construct or validate the model directly.

In [15]:
from pydantic import ValidationError

# what is this doing?
# this is creating a PatientData instance with default values for the optional fields 
# (married, allergies, conditions) and printing it. It also demonstrates error 
# handling by trying to validate a PatientData instance with missing required 
# fields and catching the ValidationError.

print("example valid case:")
patient_with_defaults = PatientData.model_validate({
    "patient_id": 2,
    "name": "Amina",
    "age": 42,
    
})
print(patient_with_defaults)


print("example error case:")
try:
    PatientData.model_validate({"patient_id": 3, "name": "Leo"})
except ValidationError as error:


    print("Required field is missing:")
    print(error)


example valid case:
patient_id=2 name='Amina' age=42 married=False medications=None allergies=[] conditions=[]
example error case:
Required field is missing:
1 validation error for PatientData
age
  Field required [type=missing, input_value={'patient_id': 3, 'name': 'Leo'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [22]:
from pydantic import ValidationError, BaseModel, Field, EmailStr, AnyUrl, field_validator, model_validator
from typing import Optional, List, Dict, Union, Tuple, Set, FrozenSet, Literal, Annotated


class PatientProfile(BaseModel):
    name: Annotated[str, Field(min_length=1, max_length=100, title="Patient Name", description="The full name of the patient", json_schema_extra={"example": "John Doe"})]
    email: Annotated[EmailStr, Field(title="Patient Email", description="The email address of the patient", json_schema_extra={"example": "john.doe@example.com"})]
    linkedin_url: Annotated[Optional[AnyUrl], Field(title="LinkedIn Profile URL", description="The LinkedIn profile URL of the patient", json_schema_extra={"example": "https://www.linkedin.com/in/johndoe"})] = None
    weight: Annotated[float, Field(gt=0, title="Patient Weight", description="The weight of the patient in kilograms", json_schema_extra={"example": 70.5})]
    married: Annotated[bool, Field(title="Marital Status", description="Indicates whether the patient is married", json_schema_extra={"example": True})]
    allergies: Annotated[List[str], Field(title="Allergies", description="A list of allergies the patient has", json_schema_extra={"example": ["Peanuts", "Shellfish"]})] = Field(default_factory=list)
    contact_details: Annotated[Dict[str, str], Field(title="Contact Details", description="A dictionary containing contact details of the patient", json_schema_extra={"example": {"phone": "123-456-7890", "address": "123 Main St, Anytown, USA"}})] = Field(default_factory=dict)
    conditions: Annotated[Set[str], Field(title="Medical Conditions", description="A set of medical conditions the patient has", json_schema_extra={"example": ["Diabetes", "Hypertension"]})] = Field(default_factory=set)

    # lets create a method for wmail field validation 
    @field_validator("email")
    @classmethod
    def validate_email(cls, value):
        if not value.endswith("@example.com"):
            raise ValueError("Email must be from the domain 'example.com'")
        return value

    
    @field_validator("name")
    @classmethod
    def transform_name(cls, value):
        return value.upper()  # Capitalize the first letter of each word in the name

    @model_validator(mode="before")
    @classmethod
    def transform_all_fields(cls, values):
        # Example of transforming all fields before validation
        return values

def add_patient_data(patient: PatientProfile):
    print(patient.name)
    print(patient.email)
    print(patient.linkedin_url)
    print(patient.weight)
    print(patient.married)
    print("Data added successfully.")


# This dictionary contains the required patient profile fields.
patient_data = {"name": "John Doe", "email": "john.doe@example.com", "weight": 70.5, "married": True}

print_patient_profile(PatientProfile.model_validate(patient_data))

JOHN DOE
john.doe@example.com
None
70.5
True
Data added successfully.


## Pydantic Field and Model Validators

Validators add custom rules beyond the types and constraints declared with `Field`. They run automatically when a model is created or validated, for example with `PatientProfile.model_validate(data)`. If a validator raises `ValueError`, Pydantic reports it as part of a `ValidationError`.

### `@field_validator`

Use a field validator to check or transform one field. The validator must return the resulting value. The default mode is `after`, so it receives the value after Pydantic has parsed and checked its type. Use `mode="before"` when a rule needs to inspect or normalize the raw input first.

In `PatientProfile`, `validate_email` restricts the already-parsed email to `@example.com`, and `transform_name` changes the name to uppercase. Both run during `PatientProfile.model_validate(patient_data)`.

### `@model_validator`

Use a model validator when a rule depends on multiple fields or the model as a whole. For `mode="after"`, define an instance method, inspect `self`, and return the instance. This is useful for cross-field rules such as requiring an end value to be later than a start value:

```python
from pydantic import BaseModel, model_validator


class DateRange(BaseModel):
    start: int
    end: int

    @model_validator(mode="after")
    def check_order(self):
        if self.end <= self.start:
            raise ValueError("end must be greater than start")
        return self
```

A `mode="before"` model validator instead receives the raw input before field validation; use it to normalize or reshape incoming data. Prefer `Field` constraints for simple single-field rules, a field validator for custom logic on one field, and a model validator for relationships between fields.

In [17]:

from pydantic import BaseModel, ValidationError, model_validator


class DateRange(BaseModel):
    start: int
    end: int

    @model_validator(mode="after")
    def check_order(self):
        if self.end <= self.start:
            raise ValueError("end must be greater than start")
        return self


valid_range = DateRange.model_validate({"start": 9, "end": 17})
print("Valid range:", valid_range)

try:
    DateRange.model_validate({"start": 17, "end": 9})
except ValidationError as error:
    print("Invalid range:")
    print(error)

Valid range: start=9 end=17
Invalid range:
1 validation error for DateRange
  Value error, end must be greater than start [type=value_error, input_value={'start': 17, 'end': 9}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## Nested Pydantic Models

A model field can itself use another `BaseModel`. This lets you represent structured data as reusable pieces, such as a patient with an address or an order with line items.

When `model_validate()` receives a nested dictionary, Pydantic validates each nested field and creates the inner model automatically. You can then access nested values with attributes (for example, `patient.address.city`). Calling `model_dump()` converts the whole structure back to nested dictionaries.

Use the nested model's field types and constraints to validate its data, and use `list[SomeModel]` when a field contains multiple nested objects.

In [24]:
from pydantic import BaseModel, Field, ValidationError


class MailingAddress(BaseModel):
    street: str = Field(min_length=1)
    city: str = Field(min_length=1)
    postal_code: str = Field(min_length=4)


class PatientWithAddress(BaseModel):
    name: str
    address: MailingAddress


patient_payload = {
    "name": "Amina",
    "address": {
        "street": "12 Oak Street",
        "city": "Oslo",
        "postal_code": "0150",
    },
}

patient = PatientWithAddress.model_validate(patient_payload)
print("Nested model:", patient.address)
print("City:", patient.address.city)
print("As dictionaries:", patient.model_dump())

try:
    PatientWithAddress.model_validate({
        "name": "Amina",
        "address": {"street": "12 Oak Street", "city": "Oslo", "postal_code": ""},
    })
except ValidationError as error:
    print("Invalid nested address:")
    print(error)

Nested model: street='12 Oak Street' city='Oslo' postal_code='0150'
City: Oslo
As dictionaries: {'name': 'Amina', 'address': {'street': '12 Oak Street', 'city': 'Oslo', 'postal_code': '0150'}}
Invalid nested address:
1 validation error for PatientWithAddress
address.postal_code
  String should have at least 4 characters [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


## Pydantic Serialization

Serialization converts a validated model into a format for APIs, storage, or logging.

- `model_dump()` returns a Python dictionary. Nested models become nested dictionaries; Python-native values such as sets remain native Python values.
- `model_dump_json()` returns a JSON string. Values that JSON does not support directly, such as sets, are converted to JSON-compatible forms.
- Use `exclude_none=True` to omit fields whose value is `None`, and `exclude_unset=True` to omit fields that were not supplied and are still using defaults. Use `include=` or `exclude=` to select fields.

These options affect the serialized output; they do not change the model itself. For JSON text, use `model_dump_json()` rather than calling `str()` on the model.

In [25]:
from pydantic import BaseModel


class SerializationAddress(BaseModel):
    city: str
    country: str


class SerializationExample(BaseModel):
    name: str
    nickname: str | None = None
    address: SerializationAddress


record = SerializationExample.model_validate({
    "name": "Amina",
    "nickname": None,
    "address": {"city": "Oslo", "country": "Norway"},
})

print("Python dictionary:", record.model_dump())
print("Omitting None fields:", record.model_dump(exclude_none=True))
print("JSON text:")
print(record.model_dump_json(indent=2))

record_without_nickname = SerializationExample.model_validate({
    "name": "Leo",
    "address": {"city": "Paris", "country": "France"},
})
print("Omitting fields not supplied:", record_without_nickname.model_dump(exclude_unset=True))

Python dictionary: {'name': 'Amina', 'nickname': None, 'address': {'city': 'Oslo', 'country': 'Norway'}}
Omitting None fields: {'name': 'Amina', 'address': {'city': 'Oslo', 'country': 'Norway'}}
JSON text:
{
  "name": "Amina",
  "nickname": null,
  "address": {
    "city": "Oslo",
    "country": "Norway"
  }
}
Omitting fields not supplied: {'name': 'Leo', 'address': {'city': 'Paris', 'country': 'France'}}
